# Physics-Informed Diffusion-PINN MRI Reconstruction
## Notebook 01 — Environment Setup & Official NYU FastMRI Data Setup

This reproducible notebook verifies system hardware (GPU/CUDA), system software (Python, Git), GitHub repository access, project dependencies, official NYU fastMRI signed URL access (via Kaggle Secrets / environment variables), storage-efficient subset extraction, and HDF5 data integrity inspection.

In [ ]:
# SECTION 1 — Project Information
PROJECT_NAME = "mri-diffusion-pinn"
REPO_DEFAULT_URL = "https://github.com/Siva05-code/mri-diffusion-pinn.git"

print("========================================")
print("PROJECT INFORMATION")
print("========================================")
print(f"Project Name       : {PROJECT_NAME}")
print(f"Repository URL     : {REPO_DEFAULT_URL}")
print("Dataset            : Official NYU fastMRI (Single-Coil Knee)")
print("Framework          : PyTorch Physics-Informed Diffusion-PINN")
print("========================================")

In [ ]:
# SECTION 2 — System Information
import os
import platform

print("========================================")
print("SYSTEM INFORMATION")
print("========================================")
print(f"Operating System   : {platform.system()} {platform.release()}")
print(f"Platform           : {platform.platform()}")
print(f"Working Directory  : {os.getcwd()}")
print(f"Inside Kaggle      : {os.path.exists('/kaggle')}")
print("========================================")

In [ ]:
# SECTION 3 — Python Information
import sys

print("========================================")
print("PYTHON INFORMATION")
print("========================================")
print(f"Python Executable  : {sys.executable}")
print(f"Python Version     : {sys.version.split()[0]}")
print("========================================")

In [ ]:
# SECTION 4 — PyTorch & GPU Verification
import torch

cuda_ok = torch.cuda.is_available()
print("========================================")
print("PYTORCH & GPU VERIFICATION")
print("========================================")
print(f"PyTorch Version    : {torch.__version__}")
print(f"CUDA Available     : {cuda_ok}")
if cuda_ok:
    print(f"CUDA Version       : {torch.version.cuda}")
    print(f"GPU Name           : {torch.cuda.get_device_name(0)}")
else:
    print("GPU Status         : No GPU detected (CPU mode)")
print("========================================")

In [ ]:
# SECTION 5 — FastMRI Signed URL Secret Verification (Secure)
import os

print("========================================")
print("OFFICIAL FASTMRI CREDENTIALS CHECK")
print("========================================")

# Check Kaggle Secrets or environment variables safely without printing values
has_val_secret = False
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    if user_secrets.get_secret("FASTMRI_VAL_URL"):
        has_val_secret = True
except Exception:
    pass

has_val_env = bool(os.environ.get("FASTMRI_VAL_URL") or os.environ.get("FASTMRI_URL"))
val_access_detected = has_val_secret or has_val_env

print(f"Validation Secret Detected : {val_access_detected}")
if not val_access_detected:
    print("NOTE: Set 'FASTMRI_VAL_URL' in Kaggle Secrets (Add-ons -> Secrets)")
    print("      or via environment variable export FASTMRI_VAL_URL='<signed-url>'")
print("========================================")

In [ ]:
# SECTION 6 — Load Configuration
import yaml
from pathlib import Path

print("========================================")
print("DATASET CONFIGURATION")
print("========================================")

config_path = "configs/base.yaml"
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

val_dir = Path(cfg.get("data", {}).get("val_dir", "/kaggle/working/data/fastmri/knee_singlecoil_val"))
num_val_files = cfg.get("data", {}).get("num_val_files", 10)

print(f"Dataset Source     : {cfg.get('dataset', {}).get('source')}")
print(f"Modality           : {cfg.get('dataset', {}).get('modality')}")
print(f"Target Val Dir     : {val_dir}")
print(f"Dev Subset Size    : {num_val_files} files")
print("========================================")

In [ ]:
# SECTION 7 — Run Storage-Efficient FastMRI Downloader
import subprocess

print("========================================")
print("DATASET SUBSET EXTRACTION")
print("========================================")

# Run streaming selective extraction
cmd = [sys.executable, "-m", "src.data.download_fastmri", "--split", "val", "--max-files", str(num_val_files)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("Download Warnings/Stderr:", res.stderr)
print("========================================")

In [ ]:
# SECTION 8 — Inspect Downloaded Official HDF5 Structure
import h5py

print("========================================")
print("HDF5 DATASET INTEGRITY INSPECTION")
print("========================================")

h5_files = sorted(list(val_dir.glob("*.h5")) + list(val_dir.glob("*.hdf5")))
if h5_files:
    sample_file = h5_files[0]
    print(f"Inspecting Sample HDF5: {sample_file.name}")
    with h5py.File(sample_file, "r") as hf:
        print("HDF5 Dataset Keys:", list(hf.keys()))
        for k in hf.keys():
            item = hf[k]
            if isinstance(item, h5py.Dataset):
                print(f"  Key '{k}': shape={item.shape}, dtype={item.dtype}")
        print("Header Attributes:")
        for ak, av in hf.attrs.items():
            print(f"  {ak}: {av}")
else:
    print(f"No .h5 files found in {val_dir} yet. Supply FASTMRI_VAL_URL and run downloader.")
print("========================================")

In [ ]:
# SECTION 9 — Final Environment Report
print("========================================")
print("OFFICIAL FASTMRI ENVIRONMENT READY")
print("========================================")
print(f"Python              : OK ({sys.version.split()[0]})")
print(f"PyTorch             : OK ({torch.__version__})")
print(f"GPU Acceleration    : {'OK (' + torch.cuda.get_device_name(0) + ')' if cuda_ok else 'CPU Mode'}")
print(f"Dataset Access      : {'CONFIGURED' if val_access_detected else 'PENDING SECRET'}")
print(f"HDF5 Files Available: {len(h5_files) if 'h5_files' in locals() else 0}")
print("----------------------------------------")
print("Next Step:")
print("  Run 02_kspace_inspection.ipynb")
print("========================================")